In [1]:
import os
import pandas as pd
from pyproj import Transformer

In [2]:
# file paths
home = '/store/carroll/sbgplants/'
ref = os.path.join(home, 'schema')
raw = os.path.join(home, 'data', 'raw')

doi = os.path.join(raw, '10.15485.1618130') # Locations, metadata, and species cover from field sampling survey associated with NEON AOP survey, East River, CO 2018

out_folder = os.path.join(home, 'data', 'out_csv')

table = 'plot_event_metadata'

In [3]:
# load schema and dtype
schema = pd.read_csv(os.path.join(ref, 'sbgplants-schema.csv'))
data_types = pd.read_csv(os.path.join(ref, 'data-types.csv'))

# view relevant schema
schema = schema[schema.table_name==table]
schema

,table_name,column_name,data_type
47,plot_event_metadata,plot_name,character
48,plot_event_metadata,plot_type,character
49,plot_event_metadata,team,character
50,plot_event_metadata,collection_date,date
51,plot_event_metadata,latitude,double precision
52,plot_event_metadata,longitude,double precision
53,plot_event_metadata,gps_plot_orientation,character
54,plot_event_metadata,gps_accuracy_approx,double precision
55,plot_event_metadata,fractional_cover_method,USER-DEFINED
56,plot_event_metadata,plot_cover_photos,boolean


In [4]:
# view relevant data-types info
# for now we can't do this automatically by column_name, because column_name is not explicitly linked to enum_type in any way yet

data_types = data_types[data_types['Enum Type']=='FRACTIONAL_method']
data_types

,Schema,Enum Type,Enum Value
6,sbgplants,FRACTIONAL_method,Line-intercept-transect
7,sbgplants,FRACTIONAL_method,Point
8,sbgplants,FRACTIONAL_method,Quadrat


In [5]:
# load relevant output tables
plot = pd.read_csv(os.path.join(out_folder, 'plot.csv'))
plot

,plot_name,plot_type,campaign
0,001-ER18,Plot,East River 2018
1,002-ER18,Plot,East River 2018
2,003-ER18,Plot,East River 2018
3,004-ER18,Plot,East River 2018
4,005-ER18,Plot,East River 2018
...,...,...,...
472,474-ER18,Individual,East River 2018
473,475-ER18,Individual,East River 2018
474,476-ER18,Individual,East River 2018
475,477-ER18,Individual,East River 2018


In [14]:
# load, prepare relevant raw tables
sample_site = pd.read_csv(os.path.join(doi, 'sample_site.csv'))

# fix mistake in raw data - lat, lon column names switched. To be fixed in ESS-DIVE
sample_site = sample_site.rename(columns={
    'Longitude': 'latitude',
    'Latitude': 'longitude'
})

# format collection date
sample_site['collection_date'] = pd.to_datetime(sample_site[['Year', 'Month', 'Day']])

# map gps_accuracy_approx
gps_acc = {
    'RTK': 0.01,
    'TrimbleGeoXT': 0.5
}
sample_site['GPS_source'] = sample_site['GPS_source'].map(gps_acc)

# map fc method
fc_method = {
    'Meadow': 'Quadrat',
    'Tree': 'Visual', ## update this to match an existing ENUM VALUE or add a new ENUM VALUE - ask Dana
    'Shrub': 'Visual'
}
sample_site['fractional_cover_method'] = sample_site['VegetationType'].map(fc_method)

# map floristic survey
floristic_survey = {
    'Meadow': 1,
    'Tree': 0,
    'Shrub': 0
}
sample_site['floristic_survey'] = sample_site['VegetationType'].map(floristic_survey)

# reproject plot center coordinates to crs used for flightlines
transformer = Transformer.from_crs('EPSG:4326', 'EPSG:32613', always_xy=True)
sample_site['longitude'], sample_site['latitude'] = transformer.transform(sample_site['longitude'].values, sample_site['latitude'].values)

sample_site

,SamplingArea,Campaign,SampleSiteCode,Month,Day,Year,latitude,longitude,EPSG,GPS_source,...,VegetationType,FieldVegHeightMax_cm,FieldVegHeightMedian_cm,SoilMoisture_%_1,SoilMoisture_%_2,SoilMoisture_%_3,Foliar_IGSN,collection_date,fractional_cover_method,floristic_survey
0,RM,ER18,001-ER18,6,14,2018,4.313906e+06,327909.504290,4326.0,0.01,...,Meadow,21.0,13.0,6.0,6.0,7.0,IER18005A,2018-06-14,Quadrat,1
1,RM,ER18,002-ER18,6,14,2018,4.313913e+06,327909.473751,4326.0,0.01,...,Meadow,42.0,26.0,4.0,6.0,4.0,IER18005B,2018-06-14,Quadrat,1
2,RM,ER18,003-ER18,6,14,2018,4.313922e+06,327904.468056,4326.0,0.01,...,Meadow,31.0,17.0,6.0,7.0,5.0,IER18005C,2018-06-14,Quadrat,1
3,RM,ER18,004-ER18,6,14,2018,4.313934e+06,327909.497829,4326.0,0.01,...,Meadow,55.0,32.0,5.0,7.0,8.0,IER18005D,2018-06-14,Quadrat,1
4,RM,ER18,005-ER18,6,14,2018,4.313932e+06,327898.540633,4326.0,0.01,...,Meadow,59.0,14.0,7.0,13.0,12.0,IER18005E,2018-06-14,Quadrat,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
472,GS,ER18,474-ER18,7,30,2018,4.304137e+06,322828.406631,4326.0,0.50,...,Tree,NaN,NaN,NaN,NaN,NaN,IER180056,2018-07-30,Visual,0
473,GS,ER18,475-ER18,7,30,2018,4.304106e+06,322865.138959,4326.0,0.50,...,Tree,NaN,NaN,NaN,NaN,NaN,IER180057,2018-07-30,Visual,0
474,GS,ER18,476-ER18,7,30,2018,4.304090e+06,322866.596029,4326.0,0.50,...,Tree,NaN,NaN,NaN,NaN,NaN,IER180058,2018-07-30,Visual,0
475,GS,ER18,477-ER18,7,30,2018,4.303914e+06,322893.906595,4326.0,0.50,...,Tree,NaN,NaN,NaN,NaN,NaN,IER180049,2018-07-30,Visual,0


In [24]:
# prepare & populate out table
out_table = pd.DataFrame(columns=schema['column_name'].unique())

# pull directly from another existing output table where possible
out_table['plot_name'] = plot.plot_name
out_table['plot_type'] = plot.plot_type

# fixed over the whole table
out_table['gps_plot_orientation'] = 'Center'
out_table['plot_cover_photos'] = 0 # get rid of this field entirely? what does it mean in the context of this database

for idx, row in out_table.iterrows():
    plot_ = row['plot_name']
    out_table.loc[out_table['plot_name']==plot_, 'collection_date'] = sample_site.loc[sample_site['SampleSiteCode']==plot_, 'collection_date']
    out_table['collection_date'] = pd.to_datetime(out_table['collection_date'])
    out_table.loc[out_table['plot_name']==plot_, 'latitude'] = sample_site.loc[sample_site['SampleSiteCode']==plot_, 'latitude']
    out_table.loc[out_table['plot_name']==plot_, 'longitude'] = sample_site.loc[sample_site['SampleSiteCode']==plot_, 'longitude']
    out_table.loc[out_table['plot_name']==plot_, 'gps_accuracy_approx'] = sample_site.loc[sample_site['SampleSiteCode']==plot_, 'GPS_source']
    out_table.loc[out_table['plot_name']==plot_, 'fractional_cover_method'] = sample_site.loc[sample_site['SampleSiteCode']==plot_, 'fractional_cover_method']
    out_table.loc[out_table['plot_name']==plot_, 'floristic_survey'] = sample_site.loc[sample_site['SampleSiteCode']==plot_, 'floristic_survey']

out_table

,plot_name,plot_type,team,collection_date,latitude,longitude,gps_plot_orientation,gps_accuracy_approx,fractional_cover_method,plot_cover_photos,floristic_survey,notes,raster_plot_event_id,plot_event_metadata_id
0,001-ER18,Plot,NaN,2018-06-14,4313905.998025,327909.50429,Center,0.01,Quadrat,0,1,NaN,NaN,NaN
1,002-ER18,Plot,NaN,2018-06-14,4313912.549882,327909.473751,Center,0.01,Quadrat,0,1,NaN,NaN,NaN
2,003-ER18,Plot,NaN,2018-06-14,4313921.541996,327904.468056,Center,0.01,Quadrat,0,1,NaN,NaN,NaN
3,004-ER18,Plot,NaN,2018-06-14,4313933.535375,327909.497829,Center,0.01,Quadrat,0,1,NaN,NaN,NaN
4,005-ER18,Plot,NaN,2018-06-14,4313931.997694,327898.540633,Center,0.01,Quadrat,0,1,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
472,474-ER18,Individual,NaN,2018-07-30,4304136.95269,322828.406631,Center,0.5,Visual,0,0,NaN,NaN,NaN
473,475-ER18,Individual,NaN,2018-07-30,4304106.483359,322865.138959,Center,0.5,Visual,0,0,NaN,NaN,NaN
474,476-ER18,Individual,NaN,2018-07-30,4304090.127996,322866.596029,Center,0.5,Visual,0,0,NaN,NaN,NaN
475,477-ER18,Individual,NaN,2018-07-30,4303913.964117,322893.906595,Center,0.5,Visual,0,0,NaN,NaN,NaN


In [26]:
# confirm final column data types

print(out_table.dtypes)

# adjust as necessary
out_table['latitude'] = out_table['latitude'].astype(float)
out_table['longitude'] = out_table['longitude'].astype(float)
out_table['gps_accuracy_approx'] = out_table['gps_accuracy_approx'].astype(float)

out_table['plot_cover_photos'] = out_table['plot_cover_photos'].astype(bool)
out_table['floristic_survey'] = out_table['floristic_survey'].astype(bool)

out_table.dtypes

plot_name                          object
plot_type                          object
team                               object
collection_date            datetime64[ns]
latitude                           object
longitude                          object
gps_plot_orientation               object
gps_accuracy_approx                object
fractional_cover_method            object
plot_cover_photos                    bool
floristic_survey                     bool
notes                              object
raster_plot_event_id               object
plot_event_metadata_id             object
dtype: object


plot_name                          object
plot_type                          object
team                               object
collection_date            datetime64[ns]
latitude                          float64
longitude                         float64
gps_plot_orientation               object
gps_accuracy_approx               float64
fractional_cover_method            object
plot_cover_photos                    bool
floristic_survey                     bool
notes                              object
raster_plot_event_id               object
plot_event_metadata_id             object
dtype: object

In [27]:
# export table
fp_out = os.path.join(out_folder, f'{table}.csv')
out_table.to_csv(fp_out, index=False)